In [1]:
import os
import io
from google.cloud import vision
import torch

In [3]:
DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [3]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'GOOGLE_VISION_API.json'
client = vision.ImageAnnotatorClient()

In [4]:
def get_full_vision_anal(img_pth):
    try:
        with open(img_pth, 'rb') as image_file:
            content = image_file.read()
        image = vision.Image(content=content)
        features = [
            {'type_': vision.Feature.Type.DOCUMENT_TEXT_DETECTION},
            {'type_': vision.Feature.Type.SAFE_SEARCH_DETECTION},
            {'type_': vision.Feature.Type.LANDMARK_DETECTION},
            {'type_': vision.Feature.Type.LOGO_DETECTION},
            {'type_': vision.Feature.Type.WEB_DETECTION}
        ]
        response = client.annotate_image({'image': image, 'features': features})
        return response, None
    except Exception as e:
        return None, str(e)

In [5]:
def get_in_image_anal(img_pth):
    response, error = get_full_vision_anal(img_pth)
    if error:
        return {'error': error}
    report = {}
    # OCR
    if response.full_text_annotation:
        report['Extracted Text'] = response.full_text_annotation.text
    # SAFE SEARCH
    if response.safe_search_annotation:
        safe = response.safe_search_annotation
        report['Safe Search'] = {
            'adult': vision.Likelihood(safe.adult).name,
            'violence': vision.Likelihood(safe.violence).name,
            'spoof': vision.Likelihood(safe.spoof).name
        }
    # LANDMARKS AND LOGOS
    entities = []
    if response.landmark_annotations:
        for landmark in response.landmark_annotations:
            entities.append(f'Landmark: {landmark.description}')
    if response.logo_annotations:
        for logo in response.logo_annotations:
            entities.append(f'Logo: {logo.description}')
    if entities:
        report['Identified Entities'] = entities
    return report

In [6]:
def rev_img_search(img_pth):
    response, error = get_full_vision_anal(img_pth)
    if error:
        return {'error': error}
    report = {}
    if response.web_detection and response.web_detection.pages_with_matching_images:
        matches = []
        for i in response.web_detection.pages_with_matching_images[:5]:
            matches.append({'title': i.page_title, 'url': i.url})
        report['Reverse Image Matches'] = matches
    return report

In [8]:
image_file = r"E:\GENAI H2S\Sharbat_Gula.jpg"

print("--- Analyzing In-Image Content ---")
in_image_report = get_in_image_anal(image_file)
print(in_image_report)

print("\n--- Performing Reverse Image Search ---")
reverse_search_report = rev_img_search(image_file)
print(reverse_search_report)

--- Analyzing In-Image Content ---
{'error': '403 This API method requires billing to be enabled. Please enable billing on project #52291968046 by visiting https://console.developers.google.com/billing/enable?project=52291968046 then retry. If you enabled billing for this project recently, wait a few minutes for the action to propagate to our systems and retry. [reason: "BILLING_DISABLED"\ndomain: "googleapis.com"\nmetadata {\n  key: "service"\n  value: "vision.googleapis.com"\n}\nmetadata {\n  key: "containerInfo"\n  value: "52291968046"\n}\nmetadata {\n  key: "consumer"\n  value: "projects/52291968046"\n}\n, locale: "en-US"\nmessage: "This API method requires billing to be enabled. Please enable billing on project #52291968046 by visiting https://console.developers.google.com/billing/enable?project=52291968046 then retry. If you enabled billing for this project recently, wait a few minutes for the action to propagate to our systems and retry."\n, links {\n  description: "Google devel